# Notebook 02 — Feature Engineering
**Project:** Fraud Detection ML — Sparkov Simulated Transactions  
**Input:** `data/raw/fraudTrain.csv`  
**Output:** `data/processed/X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`

Raw data doesn't go directly into a model. The Sparkov dataset has names, addresses, and timestamps — none of which an XGBoost model can use directly. Feature engineering is the process of turning those raw fields into numbers that capture the **fraud signals** we know matter from domain experience.

Every feature in this notebook was chosen because it has a real operational meaning in fraud detection.

## Cell 1 — Imports and Load Data
We load the raw training file and immediately parse the transaction timestamp as a proper datetime object. Every time-based feature we engineer depends on this step being done correctly.

**Libraries used:**
- `pandas` — data manipulation
- `numpy` — numerical operations
- `sklearn.preprocessing.LabelEncoder` — converts category strings to integers
- `pathlib.Path` — clean cross-platform file paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

# Path constants 
DATA_RAW       = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Load raw training data 
df = pd.read_csv(DATA_RAW / 'fraudTrain.csv')

# Parse transaction timestamp immediately — all time features depend on this
df['trans_datetime'] = pd.to_datetime(df['trans_date_trans_time'])

# Confirm load and inspect data
print(f'Shape: {df.shape}')
print(f'Dtypes:\n{df.dtypes}')
print(f'\nFirst 3 rows:')
df.head(3)

Shape: (1296675, 24)
Dtypes:
Unnamed: 0                        int64
trans_date_trans_time            object
cc_num                            int64
merchant                         object
category                         object
amt                             float64
first                            object
last                             object
gender                           object
street                           object
city                             object
state                            object
zip                               int64
lat                             float64
long                            float64
city_pop                          int64
job                              object
dob                              object
trans_num                        object
unix_time                         int64
merch_lat                       float64
merch_long                      float64
is_fraud                          int64
trans_datetime           datetime64[ns]
dtype: obje

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,trans_datetime
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0,2019-01-01 00:00:18
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0,2019-01-01 00:00:44
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0,2019-01-01 00:00:51


## Cell 2 — Feature: Cardholder Age
The `dob` (date of birth) column gives us the cardholder's age at the time of the transaction. We compute this as a single integer: `transaction year − birth year`.

**Why age matters as a fraud signal:**  
Fraudsters who take over accounts or use stolen card details don't always target a specific demographic — but the patterns differ by age group. Very young cardholders (students) and elderly cardholders are statistically more targeted by certain fraud types. Age also interacts with spending patterns: a $3,000 transaction from a 22-year-old is a different risk profile than from a 55-year-old. In a real Stripe ops context, age is used to calibrate the prior probability of certain transaction types being genuine.

In [2]:
# Parse date of birth and calculate age at time of transaction
df['dob_parsed'] = pd.to_datetime(df['dob'])
df['age'] = df['trans_datetime'].dt.year - df['dob_parsed'].dt.year

# Verify: print distribution and check for any unrealistic values
print(f'Age feature created.')
print(f'Min age: {df["age"].min()}')
print(f'Max age: {df["age"].max()}')
print(f'Mean age: {df["age"].mean():.1f}')
print(f'\nAge by fraud class:')
print(df.groupby('is_fraud')['age'].describe().round(2))

Age feature created.
Min age: 14
Max age: 96
Mean age: 46.0

Age by fraud class:
              count   mean    std   min   25%   50%   75%   max
is_fraud                                                       
0         1289169.0  46.01  17.37  14.0  33.0  44.0  57.0  96.0
1            7506.0  48.87  18.86  14.0  33.0  48.0  61.0  94.0


## Cell 3 — Features: Time-Based Signals
We extract four features from the transaction timestamp:

| Feature | Description |
|---------|-------------|
| `hour_of_day` | 0–23, the hour the transaction occurred |
| `day_of_week` | 0=Monday, 6=Sunday |
| `is_weekend` | 1 if Saturday or Sunday, 0 otherwise |
| `is_night` | 1 if hour is between 22:00 and 06:00 |

**Why time patterns matter in fraud ops:**  
In EDA (notebook 01) we saw fraud spike sharply between midnight and 4am. This is a well-known pattern in real fraud operations: legitimate cardholders are asleep, but automated fraud scripts are not. At Stripe, `hour_of_day` is one of the first fields used in velocity rules. The `is_night` binary flag makes this signal directly usable by the model without it needing to learn the hour pattern from scratch.

In [3]:
# Extract time components from the parsed datetime
df['hour_of_day']  = df['trans_datetime'].dt.hour
df['day_of_week']  = df['trans_datetime'].dt.dayofweek  # 0=Monday, 6=Sunday
df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
df['is_night']     = ((df['hour_of_day'] >= 22) | (df['hour_of_day'] <= 6)).astype(int)

# Verify: check fraud rate in night vs day transactions
print('Time features created.')
print(f'\nFraud rate — daytime: {df[df["is_night"]==0]["is_fraud"].mean()*100:.4f}%')
print(f'Fraud rate — night:   {df[df["is_night"]==1]["is_fraud"].mean()*100:.4f}%')
print(f'\nSample of time features:')
df[['trans_datetime', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_night']].head(5)

Time features created.

Fraud rate — daytime: 0.1153%
Fraud rate — night:   1.5092%

Sample of time features:


,trans_datetime,hour_of_day,day_of_week,is_weekend,is_night
0,2019-01-01 00:00:18,0,1,0,1
1,2019-01-01 00:00:44,0,1,0,1
2,2019-01-01 00:00:51,0,1,0,1
3,2019-01-01 00:01:16,0,1,0,1
4,2019-01-01 00:03:06,0,1,0,1


## Cell 4 — Feature: Geographic Distance (Haversine)
We calculate the straight-line distance in kilometres between the cardholder's registered home location (`lat`, `long`) and the merchant location (`merch_lat`, `merch_long`).

**What is the Haversine formula?**  
The Earth is a sphere. Simple Pythagorean distance doesn't work for GPS coordinates because longitude lines converge towards the poles. The Haversine formula accounts for the Earth's curvature to give the true shortest-path (great-circle) distance between two points. For the distances involved in credit card fraud (tens to thousands of km), this is much more accurate than flat-earth math.

**Why geo_distance_km is a strong fraud signal:**  
A cardholder registered in Texas making a transaction at a merchant in Alaska (or in a foreign country) is suspicious. At Stripe, geographic anomaly is one of the top 5 signals used in real-time fraud scoring. A transaction 500km+ from the cardholder's home address — especially at night, especially online — is a major red flag. This is why we track `lat/long` at account creation.

In [4]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Returns distance in km between two GPS coordinate pairs."""
    R = 6371  # Earth's radius in km
    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    # Haversine formula
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# Apply to every row (vectorised — fast on 1.3M rows)
df['geo_distance_km'] = haversine_distance(
    df['lat'].values,
    df['long'].values,
    df['merch_lat'].values,
    df['merch_long'].values
)

print('geo_distance_km created.')
print(f'Min distance:  {df["geo_distance_km"].min():.2f} km')
print(f'Max distance:  {df["geo_distance_km"].max():.2f} km')
print(f'Mean distance: {df["geo_distance_km"].mean():.2f} km')
print(f'\nAvg distance — legitimate: {df[df["is_fraud"]==0]["geo_distance_km"].mean():.2f} km')
print(f'Avg distance — fraud:      {df[df["is_fraud"]==1]["geo_distance_km"].mean():.2f} km')

geo_distance_km created.
Min distance:  0.02 km
Max distance:  152.12 km
Mean distance: 76.11 km

Avg distance — legitimate: 76.11 km
Avg distance — fraud:      76.27 km


## Cell 5 — Feature: Transaction Velocity (24h)
Velocity measures how many times the same card has been used recently. We count transactions per card per calendar day using a cumulative count — this tells the model how many transactions this card has already made today by the time this transaction occurs.

**Why velocity catches card-testing fraud:**  
Card testing is one of the most common fraud patterns on payment platforms. A fraudster who steals a batch of card numbers runs small automated transactions to check which cards are still active — often dozens in a few minutes. At Stripe, a card with 5+ transactions in an hour triggers an immediate review queue. The `velocity_24h` feature captures this burst behaviour. A card that has made 0 transactions in the last 24 hours looks very different from one that has made 20.

In [5]:
# Sort by card and time so cumcount reflects chronological order
df.sort_values(['cc_num', 'trans_datetime'], inplace=True)
df.reset_index(drop=True, inplace=True)

# Count transactions per card per calendar day (proxy for 24h velocity)
# cumcount gives 0 for the first transaction, 1 for the second, etc.
df['trans_date_only'] = df['trans_datetime'].dt.date
df['velocity_24h'] = df.groupby(['cc_num', 'trans_date_only']).cumcount()

# Remove the helper column
df.drop('trans_date_only', axis=1, inplace=True)

print('velocity_24h created.')
print(f'\nVelocity stats — legitimate transactions:')
print(df[df['is_fraud']==0]['velocity_24h'].describe().round(2))
print(f'\nVelocity stats — fraud transactions:')
print(df[df['is_fraud']==1]['velocity_24h'].describe().round(2))

velocity_24h created.

Velocity stats — legitimate transactions:
count    1289169.00
mean           2.00
std            2.32
min            0.00
25%            0.00
50%            1.00
75%            3.00
max           33.00
Name: velocity_24h, dtype: float64

Velocity stats — fraud transactions:
count    7506.00
mean        2.52
std         2.19
min         0.00
25%         1.00
50%         2.00
75%         4.00
max        15.00
Name: velocity_24h, dtype: float64


## Cell 6 — Feature: Merchant Category Encoding
The `category` column contains string labels like `'shopping_net'`, `'grocery_pos'`, `'entertainment'`. Machine learning models need numbers, not strings — so we convert each category to an integer using **Label Encoding**.

**Why label encoding here and not one-hot encoding?**  
One-hot encoding creates a separate binary column for each category (14 columns for 14 categories). Label encoding creates a single integer column. For tree-based models like XGBoost, label encoding is preferred because XGBoost can find the right splits at each integer value directly — it doesn't need the binary columns that one-hot creates. One-hot encoding is more appropriate for linear models (logistic regression) that assume a specific relationship between input values.

In [6]:
# Label encode the merchant category
le = LabelEncoder()
df['category_encoded'] = le.fit_transform(df['category'])

# Show the mapping so we know what each integer means
category_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print('Category encoding mapping:')
for cat, code in sorted(category_mapping.items(), key=lambda x: x[1]):
    print(f'  {code:2d} → {cat}')

print(f'\nShape after encoding: {df.shape}')

Category encoding mapping:
   0 → entertainment
   1 → food_dining
   2 → gas_transport
   3 → grocery_net
   4 → grocery_pos
   5 → health_fitness
   6 → home
   7 → kids_pets
   8 → misc_net
   9 → misc_pos
  10 → personal_care
  11 → shopping_net
  12 → shopping_pos
  13 → travel

Shape after encoding: (1296675, 33)


## Cell 7 — Feature: Log-Transformed Amount
The raw transaction amount (`amt`) is heavily right-skewed — most transactions are small (under $100) but there are some very large outliers (over $1,000). This skewness can hurt model performance because the model is forced to work across an enormous numeric range.

**Why log transform helps for financial data:**  
The `log1p` transformation (log of amount + 1) compresses large values while keeping the relative differences meaningful. After transformation, the difference between $1 and $10 gets the same weight as the difference between $100 and $1,000 — which is much more sensible in a fraud context. We add 1 before taking the log to handle any zero-amount transactions safely (log(0) is undefined).

We keep **both** `amt` (original) and `log_amt` (transformed) — XGBoost will pick whichever gives better splits.

In [7]:
# Print raw amount statistics first
print('Raw amount (amt) statistics:')
print(f'  Min:  ${df["amt"].min():.2f}')
print(f'  Max:  ${df["amt"].max():.2f}')
print(f'  Mean: ${df["amt"].mean():.2f}')
print(f'  Std:  ${df["amt"].std():.2f}')

# Apply log1p transformation
df['log_amt'] = np.log1p(df['amt'])

print(f'\nLog-transformed amount (log_amt) statistics:')
print(f'  Min:  {df["log_amt"].min():.4f}')
print(f'  Max:  {df["log_amt"].max():.4f}')
print(f'  Mean: {df["log_amt"].mean():.4f}')
print(f'  Std:  {df["log_amt"].std():.4f}')
print(f'\nlog_amt feature created.')

Raw amount (amt) statistics:
  Min:  $1.00
  Max:  $28948.90
  Mean: $70.35
  Std:  $160.32

Log-transformed amount (log_amt) statistics:
  Min:  0.6931
  Max:  10.2733
  Mean: 3.5335
  Std:  1.2894

log_amt feature created.


## Cell 8 — Drop Unnecessary Columns
We drop columns that are either:
- **PII (personally identifiable information):** names, addresses, zip codes — not useful as model features and should not be in training data
- **Identifiers:** transaction number, card number — unique per row, useless for generalisation
- **Redundant:** raw datetime string (we have the parsed version), dob (we computed age from it), original category string (we have the encoded version)
- **Leakage risk:** `unix_time` is just another timestamp encoding — redundant

After dropping, we print the final feature list so we know exactly what goes into the model.

In [8]:
# Columns to drop: PII, identifiers, redundant raw fields
cols_to_drop = [
    'Unnamed: 0',          # row index from CSV
    'first', 'last',       # cardholder name
    'street', 'city', 'zip',  # address PII
    'trans_num',           # transaction ID — unique per row, not a feature
    'unix_time',           # redundant timestamp encoding
    'merch_zipcode' if 'merch_zipcode' in df.columns else None,
    'cc_num',              # card identifier — use velocity instead
    'merchant',            # high-cardinality string — use category_encoded instead
    'dob', 'dob_parsed',   # used to compute age — no longer needed
    'trans_date_trans_time',  # raw string — we have trans_datetime
    'trans_datetime',      # datetime object — not needed after feature extraction
    'category',            # replaced by category_encoded
    'job',                 # not a fraud signal in this dataset
    'state',               # high-cardinality — geo_distance_km captures geography
]

# Only drop columns that actually exist (safe handling)
cols_to_drop = [c for c in cols_to_drop if c is not None and c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)

print(f'Dropped {len(cols_to_drop)} columns.')
print(f'\nFinal shape: {df.shape}')
print(f'\nFinal columns ({len(df.columns)}):')
for col in df.columns:
    print(f'  {col}')

Dropped 17 columns.

Final shape: (1296675, 17)

Final columns (17):
  amt
  gender
  lat
  long
  city_pop
  merch_lat
  merch_long
  is_fraud
  age
  hour_of_day
  day_of_week
  is_weekend
  is_night
  geo_distance_km
  velocity_24h
  category_encoded
  log_amt


## Cell 9 — Time-Based Train/Test Split **CRITICAL**
We split the data into training and test sets using **time order**, not random shuffling.

**Why time-based split — not random?**  
In real fraud detection, a model is always trained on *past* data and deployed to score *future* transactions. If we split randomly, future data leaks into the training set — the model learns patterns from transactions it should never have seen. This produces artificially inflated evaluation scores that don't reflect real-world performance.

Time-based split simulates production reality: train on the first 80% of transactions (chronologically), test on the last 20%. This is sometimes called a "walk-forward" or "temporal" split.

**Rule: SMOTE is NOT applied here.** SMOTE (oversampling) happens in notebook 04, on the training set only, immediately before model training. Applying it here would leak synthetic fraud samples into our evaluation.

In [9]:
# The data was already sorted by trans_datetime in Cell 5
# Confirm sort order is correct by re-loading index
df.reset_index(drop=True, inplace=True)


# Encode gender at the source (F=0, M=1) so all downstream
# notebooks receive int64 — no re-encoding needed in notebooks 04/04b
if 'gender' in df.columns and df['gender'].dtype == object:
    df['gender'] = df['gender'].map({'F': 0, 'M': 1}).fillna(0).astype(int)

# Split at 80% of sorted rows 
split_idx = int(len(df) * 0.80)
train_df  = df.iloc[:split_idx].copy()
test_df   = df.iloc[split_idx:].copy()

# Separate features and target 
TARGET = 'is_fraud'
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_test  = test_df.drop(columns=[TARGET])
y_test  = test_df[TARGET]

# Print split confirmation 
print(f'Split index: row {split_idx:,} of {len(df):,}')
print(f'\nX_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'\nFraud rate in train: {y_train.mean()*100:.4f}%')
print(f'Fraud rate in test:  {y_test.mean()*100:.4f}%')

# Save to data/processed/ 
X_train.to_csv(DATA_PROCESSED / 'X_train.csv', index=False)
X_test.to_csv(DATA_PROCESSED  / 'X_test.csv',  index=False)
y_train.to_csv(DATA_PROCESSED / 'y_train.csv', index=False, header=True)
y_test.to_csv(DATA_PROCESSED  / 'y_test.csv',  index=False, header=True)

print(f'\nSaved to data/processed/:')
print('  X_train.csv, X_test.csv, y_train.csv, y_test.csv')

Split index: row 1,037,340 of 1,296,675

X_train shape: (1037340, 16)
X_test shape:  (259335, 16)

Fraud rate in train: 0.5740%
Fraud rate in test:  0.5985%

Saved to data/processed/:
  X_train.csv, X_test.csv, y_train.csv, y_test.csv


## Cell 10 — Feature Engineering Summary
A final overview of everything we built in this notebook — the inputs to notebook 04 (model training).

In [10]:
# List all engineered features
engineered = ['age', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_night',
              'geo_distance_km', 'velocity_24h', 'category_encoded', 'log_amt']
original   = [c for c in X_train.columns if c not in engineered]

print('=' * 55)
print('FEATURE ENGINEERING SUMMARY')
print('=' * 55)
print(f'Total features:          {len(X_train.columns)}')
print(f'Engineered features:     {len(engineered)}')
print(f'Original features kept:  {len(original)}')
print(f'\nTraining set:  {X_train.shape[0]:>10,} rows  |  fraud rate: {y_train.mean()*100:.4f}%')
print(f'Test set:      {X_test.shape[0]:>10,} rows  |  fraud rate: {y_test.mean()*100:.4f}%')
print(f'\nEngineered features:')
for f in engineered:
    print(f'  ✓ {f}')
print(f'\nOriginal features retained:')
for f in original:
    print(f'  · {f}')
print('=' * 55)
print('Next step: notebook 04 — XGBoost + SMOTE + SHAP + MLflow')

FEATURE ENGINEERING SUMMARY
Total features:          16
Engineered features:     9
Original features kept:  7

Training set:   1,037,340 rows  |  fraud rate: 0.5740%
Test set:         259,335 rows  |  fraud rate: 0.5985%

Engineered features:
  ✓ age
  ✓ hour_of_day
  ✓ day_of_week
  ✓ is_weekend
  ✓ is_night
  ✓ geo_distance_km
  ✓ velocity_24h
  ✓ category_encoded
  ✓ log_amt

Original features retained:
  · amt
  · gender
  · lat
  · long
  · city_pop
  · merch_lat
  · merch_long
Next step: notebook 04 — XGBoost + SMOTE + SHAP + MLflow


## What's Next
The four CSV files saved to `data/processed/` are the direct inputs to notebook 04 (model training). Notebook 03 (FinBERT embeddings, run in Google Colab) generates additional embedding features that can be merged in before model training if available.

**Key things notebook 04 will do that we deliberately left out here:**
- Apply **SMOTE** to `X_train` / `y_train` only — to handle the 0.58% class imbalance
- Train **XGBoost** on the SMOTE-balanced training set
- Evaluate using **Precision-Recall AUC** on the untouched test set
- Generate **SHAP values** to explain which features drive each prediction
- Log everything to **MLflow** for experiment tracking